In [ ]:
!pip install kneed

In [ ]:
import numpy as np
from scipy.spatial.distance import pdist, squareform
import scipy.io
import pandas as pd
from kneed import KneeLocator
import pandas as pd
from sklearn.metrics import roc_auc_score
import h5py

# # #Open the v7.3 .mat file using h5py
# with h5py.File('http.mat', 'r') as f:
#     data = f['X'][:]
#     datas = np.array(data).T

#     datay = np.array(f['y']).T
#     y = datay.flatten().astype(int)

dataR = scipy.io.loadmat('cardio.mat')
dataX = dataR['X']
datas = np.array(dataX)
datay = dataR['y']
y = np.array(datay)

# df_X = pd.read_csv('/content/glass.csv', header=None)
# df_y = pd.read_csv('/content/glassLabels.csv', header=None)
# datas = np.array(df_X)
# y = np.array(df_y)


buffer = np.empty((0, datas.shape[1]))
weights = np.array([])
limit = 100
g = 2


lof_scores = np.full(datas.shape[0], np.nan)


indices_in_buffer = []

drawValve = 0


for i, row in enumerate(datas):

  buffer = np.vstack([buffer, row.reshape(1, -1)])
  indices_in_buffer.append(i)

  if weights.size > 0:
    weights *= 0.999

  weights = np.append(weights, 1.0)

  if buffer.shape[0] >= limit:

    [L,W] = buffer.shape


    dist0 = pdist(buffer, metric='euclidean') ** 2


    dist1 = squareform(dist0)


    median_dist = np.median(dist0)


    filtered_deviations = dist0[dist0 < median_dist]


    mad2 = np.median(filtered_deviations)


    threshold = 0.5 * median_dist


    if threshold < mad2:
        g = 3
    else:
        g = 2

    averdist = np.mean(dist0)


    for ii in range(g):

      dist0 = dist0[dist0 <= averdist]


      averdist = np.mean(dist0)


    eye_matrix = np.eye(L)


    max_dist = np.max(dist1)


    dist2 = dist1 + eye_matrix * (max_dist + 1)


    A = np.zeros((L, L))


    A[dist2 <= averdist] = 1


    NN = int(np.ceil(np.mean(np.sum(A, axis=1))))


    A2 = np.argsort(dist2, axis=1)


    nn = A2[:, :NN]


    RD = np.zeros((L, NN))


    for i in range(L):
      for j_idx, j in enumerate(nn[i]):

        k_distance_j = dist2[j, nn[j, NN-1]]
        RD[i, j_idx] = max(k_distance_j, dist2[i, j])


    LRD = 1 / (np.mean(RD, axis=1))


    LOF = np.zeros(L)
    for i in range(L):
      LOF[i] = np.sum(LRD[nn[i]] / LRD[i]) / NN


    for idx, lof_value in zip(indices_in_buffer, LOF):
      lof_scores[idx] = lof_value

    num_to_remove = 1
    top_outlier_indices = np.argsort(LOF)[-num_to_remove:]
    removal_index = top_outlier_indices[0]

    sorted_lof = np.sort(LOF)
    lof_indices = np.arange(len(sorted_lof))


    diff_matrix = LOF[:, None] - LOF


    diff_matrix_no_self = diff_matrix[~np.eye(len(LOF), dtype=bool)].reshape(len(LOF), -1)


    median_differences = np.median(diff_matrix_no_self, axis=1)

    sort_m = np.sort(median_differences)
    sort_m_indices = np.arange(len(median_differences))

    knee_locator = KneeLocator(sort_m_indices, sort_m, curve="convex", direction="increasing", S=5)
    lof_threshold = sort_m[knee_locator.knee]

    num_to_remove = 1
    top_outlier_indices = np.argsort(median_differences)[-num_to_remove:]
    removal_index = top_outlier_indices[0]

    if median_differences[removal_index] > np.max(lof_threshold):

      buffer = np.delete(buffer, removal_index, axis=0)
      weights = np.delete(weights, removal_index)
      del indices_in_buffer[removal_index]
    else:
      removal_index = np.argmin(weights)

      buffer = np.delete(buffer, removal_index, axis=0)
      weights = np.delete(weights, removal_index)
      del indices_in_buffer[removal_index]

auc = roc_auc_score(y, lof_scores)
print(auc)